In [ ]:
import numpy as np 
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
messages:list[str] = "Today is the day, let start a new"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2",use_fast=True)
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"{model.generation_config.get_generation_mode()=}")

In [ ]:
from transformers.generation.logits_process import RepetitionPenaltyLogitsProcessor
from transformers.generation import GenerationConfig
# - *greedy decoding* if `num_beams=1` and `do_sample=False`
# - *multinomial sampling* if `num_beams=1` and `do_sample=True`
# - *beam-search decoding* if `num_beams>1` and `do_sample=False`
# - *beam-search multinomial sampling* if `num_beams>1` and `do_sample=True`

In [ ]:
gen_config = GenerationConfig(
    max_new_tokens=5,
    # beam-searh decoding
    do_sample=True,
    num_beams=4,
    early_stopping=True,
    top_k=10,
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
    forced_eos_token_id=tokenizer.eos_token_id,
    return_dict_in_generate=True,
    output_scores=True,
)

In [ ]:
inputs = tokenizer(messages, return_tensors="pt",truncation=True,padding=True,padding_side='right',)
inputs

In [ ]:
tokenizer.decode([8888,  318,  262, 1110,   11, 1309,  923,  257,  649])

In [ ]:
tokenizer.decode(  [ 5756,   923,   257,   649, 50256, 50256])

In [ ]:
# Pro tip: In practice, LLMs use `top_p` in the 0.9-0.95 range
# temperature=0 means gready decode
# Pro tip: In practice, LLMs use `top_k` in the 5-50 range.
rep_pen_processor = RepetitionPenaltyLogitsProcessor(penalty=1.1,prompt_ignore_length=inputs["input_ids"].shape[-1])

In [ ]:
outputs = model.generate(
    **inputs, 
    generation_config=gen_config
    )
outputs

In [ ]:
transition_scores = model.compute_transition_scores(outputs.sequences, outputs.scores, normalize_logits=True)
transition_scores

In [ ]:
input_length = 1 if model.config.is_encoder_decoder else inputs.input_ids.shape[1]
print(f'{input_length=}')

In [ ]:
outputs.sequences

In [ ]:
generated_tokens = outputs.sequences[:, input_length:]
generated_tokens

In [ ]:
[tokenizer.decode([_]) for _ in generated_tokens[0]]

In [ ]:
for tok, score in zip(generated_tokens[0], transition_scores[0]):
    # | token | token string | log probability | probability
    print(f"| {tok:5d} | {tokenizer.decode(tok):8s} | {score.numpy():.3f} | {np.exp(score.numpy()):.2%}")